In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:3' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-22 10:54:43,267]::456791383::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:3


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)

In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# Multi-seed Training
# ============================================================

# Only ONE checkpoint file is kept.
# Whenever a better validation MSE is found across all seeds,
# this same file is overwritten.
from vit import TimeSformer


best_model_path = "./save_timesformer_folder/model_best_multiseed.pth"

seed_results = []
global_best_mse = float("inf")
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(3):

    for seed in SEEDS:

        print("\n" + "=" * 80)
        print(f"Start training | seed = {seed}")
        print("=" * 80)

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_seed(seed)

        # Re-create DataLoaders for each seed so that the
        # shuffled training order is also controlled by seed.
        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Every seed starts from a newly initialized model.
        # ----------------------------------------------------
        model=TimeSformer()#num_frames=frame_nums,batchsz = batchsz
        model.to(device)

        criterion = torch.nn.MSELoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        seed_best_mse = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):

            train_avg_loss = 0
            val_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            # ------------------------------------------------
            # Train
            # ------------------------------------------------
            model.train()

            for audio, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1)
                big_five_data = big_five_data.squeeze()
                audio = audio.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                hypothesis = model(fullshot,audio)
                loss = criterion(hypothesis, big_five_data)

                loss.backward()
                optimizer.step()

                train_avg_loss += loss

            train_avg_loss = train_avg_loss / len(train_dataloader)

            # ------------------------------------------------
            # Validation
            # ------------------------------------------------
            model.eval()

            with torch.no_grad():

                for audio, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1)
                    big_five_data = big_five_data.squeeze()
                    audio = audio.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot,audio)
                    val_loss = criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))

                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_avg_loss += val_loss
                    val_avg_o_loss += val_o_loss
                    val_avg_c_loss += val_c_loss
                    val_avg_e_loss += val_e_loss
                    val_avg_a_loss += val_a_loss
                    val_avg_n_loss += val_n_loss

                val_avg_loss = val_avg_loss / len(val_dataloader)
                val_avg_o_loss = val_avg_o_loss / len(val_dataloader)
                val_avg_c_loss = val_avg_c_loss / len(val_dataloader)
                val_avg_e_loss = val_avg_e_loss / len(val_dataloader)
                val_avg_a_loss = val_avg_a_loss / len(val_dataloader)
                val_avg_n_loss = val_avg_n_loss / len(val_dataloader)

            # Convert to Python float for comparisons / logging.
            current_val_mse = val_avg_loss.item()

            # Best result for current seed: lower MSE is better.
            if current_val_mse < seed_best_mse:
                seed_best_mse = current_val_mse
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # Global best checkpoint across all seeds.
            if current_val_mse < global_best_mse:
                global_best_mse = current_val_mse
                global_best_seed = seed
                global_best_epoch = i + 1
                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "loss": current_val_mse,
                }, best_model_path)
                logger.info(
                    "Global best updated | seed: {} | epoch: {} | MSE: {:.6f}".format(
                        seed, i + 1, current_val_mse
                    )
                )

            logger.debug(
                "Seed: {} | Epoch: {} | MSE: {:.6f} | "
                "O-MSE: {:.6f} | C-MSE: {:.6f} | E-MSE: {:.6f} | "
                "A-MSE: {:.6f} | N-MSE: {:.6f}".format(
                    seed, i + 1, current_val_mse,
                    val_avg_o_loss.item(), val_avg_c_loss.item(),
                    val_avg_e_loss.item(), val_avg_a_loss.item(),
                    val_avg_n_loss.item()
                )
            )

            torch.cuda.empty_cache()

            # Early stopping based on validation MSE.
            if no_improve_count >= patience:
                logger.info(
                    "Early stopping | seed: {} | epoch: {} | "
                    "best epoch: {} | best MSE: {:.6f}".format(
                        seed, i + 1, seed_best_epoch, seed_best_mse
                    )
                )
                break

        seed_results.append({
            "seed": seed,
            "best_epoch": seed_best_epoch,
            "best_mse": seed_best_mse
        })

        print(
            f"Seed {seed} finished | "
            f"Best epoch = {seed_best_epoch} | "
            f"Best MSE = {seed_best_mse:.6f}"
        )

        del model
        del optimizer
        del criterion
        del train_dataloader
        del val_dataloader
        del generator
        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# Multi-seed Summary (MSE)
# ============================================================
all_mse = np.array(
    [result["best_mse"] for result in seed_results],
    dtype=np.float64
)
mean_mse = all_mse.mean()
std_mse = all_mse.std(ddof=1) if len(all_mse) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

for result in seed_results:
    print(
        "Seed {:>4} | Best Epoch {:>3} | MSE {:.6f}".format(
            result["seed"], result["best_epoch"], result["best_mse"]
        )
    )

print("-" * 80)
print(f"Mean MSE : {mean_mse:.6f}")
print(f"Std  MSE : {std_mse:.6f}")
print(
    f"Global best: seed={global_best_seed}, "
    f"epoch={global_best_epoch}, MSE={global_best_mse:.6f}"
)
print(f"Saved checkpoint: {best_model_path}")
print("=" * 80)



Start training | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 42 | Train 1/120: 100%|██████████| 1091/1091 [04:46<00:00,  3.81it/s]
Seed 42 | Valid 1/120: 100%|██████████| 191/191 [00:18<00:00, 10.14it/s]
[2026-09-22 11:00:00,000]::456914014::INFO::Global best updated | seed: 42 | epoch: 1 | MSE: 1.580511
[2026-09-22 11:00:00,001]::456914014::DEBUG::Seed: 42 | Epoch: 1 | MSE: 1.580511 | O-MSE: 1.463992 | C-MSE: 1.299131 | E-MSE: 1.268745 | A-MSE: 1.764771 | N-MSE: 2.105915
Seed 42 | Train 2/120: 100%|██████████| 1091/1091 [04:47<00:00,  3.80it/s]
Seed 42 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.20it/s]
[2026-09-22 11:05:11,443]::456914014::INFO::Global best updated | seed: 42 | epoch: 2 | MSE: 1.337249
[2026-09-22 11:05:11,445]::456914014::DEBUG::Seed: 42 | Epoch: 2 | MSE: 1.337249 | O-MSE: 1.122326 | C-MSE: 1.190711 | E-MSE: 1.228456 | A-MSE: 1.620838 | N-MSE: 1.523917
Seed 42 | Train 3/120: 100%|██████████| 1091/1091 [04:47<00:00,  3.79it/s]
Seed 42 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.18it/s]
[2026-09-2

Seed 42 finished | Best epoch = 2 | Best MSE = 1.337249

Start training | seed = 123
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 123 | Train 1/120: 100%|██████████| 1091/1091 [04:51<00:00,  3.75it/s]
Seed 123 | Valid 1/120: 100%|██████████| 191/191 [00:20<00:00,  9.24it/s]
[2026-09-22 12:53:41,955]::456914014::DEBUG::Seed: 123 | Epoch: 1 | MSE: 1.605912 | O-MSE: 1.311142 | C-MSE: 1.592080 | E-MSE: 1.172027 | A-MSE: 2.001859 | N-MSE: 1.952454
Seed 123 | Train 2/120: 100%|██████████| 1091/1091 [04:50<00:00,  3.75it/s]
Seed 123 | Valid 2/120: 100%|██████████| 191/191 [00:20<00:00,  9.14it/s]
[2026-09-22 12:58:55,625]::456914014::DEBUG::Seed: 123 | Epoch: 2 | MSE: 1.482577 | O-MSE: 1.271313 | C-MSE: 1.353341 | E-MSE: 1.315719 | A-MSE: 1.824763 | N-MSE: 1.647745
Seed 123 | Train 3/120: 100%|██████████| 1091/1091 [04:50<00:00,  3.75it/s]
Seed 123 | Valid 3/120: 100%|██████████| 191/191 [00:20<00:00,  9.14it/s]
[2026-09-22 13:04:09,044]::456914014::DEBUG::Seed: 123 | Epoch: 3 | MSE: 1.420373 | O-MSE: 1.178308 | C-MSE: 1.473271 | E-MSE: 1.274569 | A-MSE: 1.668455 | N-MSE: 1.507263
Seed 123 | Train 4/120: 100%|█████

Seed 123 finished | Best epoch = 3 | Best MSE = 1.420373

Start training | seed = 777
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


...

Seed 777 finished | Best epoch = 2 | Best MSE = 1.356133

Start training | seed = 1004
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 1004 | Train 1/120: 100%|██████████| 1091/1091 [04:51<00:00,  3.74it/s]
Seed 1004 | Valid 1/120: 100%|██████████| 191/191 [00:20<00:00,  9.29it/s]
[2026-09-22 16:49:13,421]::456914014::DEBUG::Seed: 1004 | Epoch: 1 | MSE: 1.514341 | O-MSE: 1.267741 | C-MSE: 1.359964 | E-MSE: 1.272672 | A-MSE: 1.785966 | N-MSE: 1.885359
Seed 1004 | Train 2/120: 100%|██████████| 1091/1091 [04:51<00:00,  3.74it/s]
Seed 1004 | Valid 2/120: 100%|██████████| 191/191 [00:20<00:00,  9.27it/s]
[2026-09-22 16:54:27,661]::456914014::DEBUG::Seed: 1004 | Epoch: 2 | MSE: 1.493993 | O-MSE: 1.261116 | C-MSE: 1.391759 | E-MSE: 1.552505 | A-MSE: 1.742514 | N-MSE: 1.522071
Seed 1004 | Train 3/120: 100%|██████████| 1091/1091 [04:52<00:00,  3.73it/s]
Seed 1004 | Valid 3/120: 100%|██████████| 191/191 [00:20<00:00,  9.22it/s]
[2026-09-22 16:59:43,021]::456914014::DEBUG::Seed: 1004 | Epoch: 3 | MSE: 1.465082 | O-MSE: 1.252637 | C-MSE: 1.436223 | E-MSE: 1.256932 | A-MSE: 1.766559 | N-MSE: 1.613056
Seed 1004 | Train 4/120: 

Seed 1004 finished | Best epoch = 4 | Best MSE = 1.401986

Start training | seed = 2024
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 2024 | Train 1/120: 100%|██████████| 1091/1091 [04:51<00:00,  3.74it/s]
Seed 2024 | Valid 1/120: 100%|██████████| 191/191 [00:21<00:00,  8.93it/s]
[2026-09-22 18:55:05,166]::456914014::DEBUG::Seed: 2024 | Epoch: 1 | MSE: 1.438660 | O-MSE: 1.363834 | C-MSE: 1.424147 | E-MSE: 1.050659 | A-MSE: 1.535063 | N-MSE: 1.819598
Seed 2024 | Train 2/120: 100%|██████████| 1091/1091 [04:51<00:00,  3.75it/s]
Seed 2024 | Valid 2/120: 100%|██████████| 191/191 [00:20<00:00,  9.10it/s]
[2026-09-22 19:00:19,299]::456914014::DEBUG::Seed: 2024 | Epoch: 2 | MSE: 1.548519 | O-MSE: 1.490956 | C-MSE: 1.673887 | E-MSE: 1.343439 | A-MSE: 1.682698 | N-MSE: 1.551615
Seed 2024 | Train 3/120: 100%|██████████| 1091/1091 [04:52<00:00,  3.73it/s]
Seed 2024 | Valid 3/120: 100%|██████████| 191/191 [00:20<00:00,  9.13it/s]
[2026-09-22 19:05:34,377]::456914014::DEBUG::Seed: 2024 | Epoch: 3 | MSE: 1.499650 | O-MSE: 1.169526 | C-MSE: 1.506325 | E-MSE: 1.256767 | A-MSE: 1.862386 | N-MSE: 1.703243
Seed 2024 | Train 4/120: 

Seed 2024 finished | Best epoch = 5 | Best MSE = 1.325410

MULTI-SEED RESULTS
Seed   42 | Best Epoch   2 | MSE 1.337249
Seed  123 | Best Epoch   3 | MSE 1.420373
Seed  777 | Best Epoch   2 | MSE 1.356133
Seed 1004 | Best Epoch   4 | MSE 1.401986
Seed 2024 | Best Epoch   5 | MSE 1.325410
--------------------------------------------------------------------------------
Mean MSE : 1.368231
Std  MSE : 0.041225
Global best: seed=2024, epoch=5, MSE=1.325410
Saved checkpoint: ./save_timesformer_folder/model_best_multiseed.pth
